In [74]:
from astropy import units as u
from astropy.constants import G, c, k_B, sigma_sb, m_p
import simple_disk as sd
from matplotlib import pyplot as plt
import disk_newton_solver as newton
import numpy as np
from disk_newton_solver import Tester
import pandas as pd
import seaborn as sns

System of equations:
Equation I

$\frac{k T_c}{\mu m_p} + \frac{4\sigma T_c^4}{3c\rho}-\left[\frac{\dot{M}\Omega^2f}{3\pi\alpha\rho}\right]^{2/3}=0$

$\frac{\partial f_I}{\partial T_c} = \frac{16\sigma T_c^3}{3c\rho} + \frac{k}{m_p\mu}$

$\frac{\partial f_I}{\partial \rho} = \frac{2(f\dot{M})^{2/3}\Omega^{4/3}}{3^{5/3}(\pi\alpha)^{2/3}\rho^{5/3}}-\frac{4T_c^4\sigma}{3c\rho^2}$

Equation II

$\rho^{2/3}(\kappa_0+\kappa_1\rho T_c^{-7/2})-\left[\frac{32\pi\sigma\Omega R^3}{9GM\dot{M}f}\right]T_c^4\left[\frac{3\pi\alpha}{\dot{M}f\Omega^2}\right]^{1/3}=0$

$\frac{\partial f_{II}}{\partial \rho} = \frac{5\kappa_1\rho+2T_c^{7/2}\kappa_0}{3T_c^{7/2}(\rho)^{1/3}}$

$\frac{\partial f_{II}}{\partial T_c} = \frac{-128\pi^{4/3}(T_cR)^{3}(\alpha\Omega)^{1/3}\sigma}{3^{5/3}GM(f\dot{M})^{4/3}}-\frac{7\kappa_1\rho^{5/3}}{2T_c^{9/2}}$

Definitions:

$f=1-\sqrt{\frac{R_*}{R}}$

$\Omega=\sqrt{\frac{GM}{R^3}}$




In [88]:
mu = 0.6 
M = 1e8 * u.Msun
R_i = 6 * G * M / c / c

R = 6.5 * G * M / c / c

M_dot = 1e23 * u.kg/u.s
alpha = 0.1

f = 1-np.sqrt(R_i/R)
Omega = np.sqrt(G*M/(R**3)).si
kappa_0 = 0.04 * u.m**2 / u.kg
kappa_1 = 0 * u.m**5 / u.kg**2 * u.K**(7/2)

def f1(T_cn, rhon):
    f_I = (k_B * T_cn / (mu * m_p) + 
       4 * sigma_sb * T_cn**4 / (3 * c * rhon) - 
       (M_dot * Omega**2 * f / (3 * np.pi * alpha * rhon))**(2/3))
    return f_I.si

def f2(T_cn, rhon):
    f_II = (rhon**(2/3) * (kappa_0 + kappa_1 * rhon * T_cn**(-7/2)) - 
        (32 * np.pi * sigma_sb * Omega * R**3 / (9 * G * M * M_dot * f)) * 
        T_cn**4 * (3 * np.pi * alpha / (M_dot * f * Omega**2))**(1/3))
    
    return f_II.si

def J(T_cn, rhon):

    df_I_dT_c = (16 * sigma_sb * T_cn**3 / (3 * c * rhon) + 
             k_B / (m_p * mu))

    df_I_drho = (2 * (f * M_dot)**(2/3) * Omega**(4/3) / 
                (3**(5/3) * (np.pi * alpha)**(2/3) * rhon**(5/3)) - 
                4 * T_cn**4 * sigma_sb / (3 * c * rhon**2))
    
    df_II_drho = ((5 * kappa_1 * rhon + 2 * T_cn**(7/2) * kappa_0) / 
              (3 * T_cn**(7/2) * rhon**(1/3)))

    df_II_dT_c = (-128 * np.pi**(4/3) * (T_cn * R)**3 * (alpha * Omega)**(1/3) * sigma_sb / 
                (3**(5/3) * G * M * (f * M_dot)**(4/3)) - 
                7 * kappa_1 * rhon**(5/3) / (2 * T_cn**(9/2)))
    J11 = df_I_dT_c.si
    J12 = df_I_drho.si
    J21 = df_II_dT_c.si
    J22 = df_II_drho.si
    return [[J11, J12],[J21, J22]]

def R_neg(T_cn, rhon):
    return [-f1(T_cn, rhon), -f2(T_cn, rhon)]

Ti = 1.4e4*u.K
rhoi = 3.1e-6 * u.kg/(u.m**3)
xvec = [Ti, rhoi]
xvec

[<Quantity 14000. K>, <Quantity 3.1e-06 kg / m3>]

In [89]:
max_iterations = 5
print(rf"T_i:{xvec[0]:.6f}   |   rho_i:{xvec[1].si:.6e}")
for i in range(0, max_iterations):
    x1n = xvec[0]
    x2n = xvec[1]
    Jn = J(x1n, x2n)
    Rn = R_neg(x1n, x2n)
    
    # de-unit
    J_matrix = np.array([[Jn[0][0].value, Jn[0][1].value],
                         [Jn[1][0].value, Jn[1][1].value]])
    R_vector = np.array([Rn[0].value, Rn[1].value])
    
    # not a problem when moving to C++, but this requires removing units
    delta_values = np.linalg.solve(a=J_matrix, b=R_vector)
    
    # put units back
    delta_0_units = Rn[0].unit / Jn[0][0].unit
    delta_1_units = Rn[1].unit / Jn[1][1].unit
    
    delta = [delta_values[0] * delta_0_units, 
             delta_values[1] * delta_1_units]
    
    xvec = [(x1n + delta[0]), (x2n + delta[1])]
    print(rf"T_{i}:{xvec[0]:.6f}   |   rho {i}:{xvec[1]:.6f}")

T_i:14000.000000 K   |   rho_i:3.100000e-06 kg / m3
T_0:989572178.486417 K   |   rho 0:-0.000001 kg / m3
T_1:nan K   |   rho 1:nan kg / m3
T_2:nan K   |   rho 2:nan kg / m3
T_3:nan K   |   rho 3:nan kg / m3
T_4:nan K   |   rho 4:nan kg / m3
